In [5]:
# Evaluate result-list lengths using the saved clustered_verses.csv file

import pandas as pd
import numpy as np



CLUSTERED_FILE = "../data/clustered_verses.csv"

# Number of verses selected from each Surah
N_VERSES_PER_SURAH = 1

# Reproducible sampling
SAMPLE_RANDOM_STATE = 42

# The saved CSV contains some verse# = 0 rows representing
# Surah headings and basmala entries.
# Set to True to exclude them.
EXCLUDE_VERSE_ZERO = True


# The app displays all other verses in the selected cluster.
# Therefore, the query verse itself is excluded from the list.
EXCLUDE_QUERY_VERSE = True



In [6]:

clustered_df = pd.read_csv(
    CLUSTERED_FILE,
    encoding="utf-8-sig"
)

clustered_df["chapter#"] = pd.to_numeric(
    clustered_df["chapter#"],
    errors="coerce"
)

clustered_df["verse#"] = pd.to_numeric(
    clustered_df["verse#"],
    errors="coerce"
)

clustered_df["cluster"] = pd.to_numeric(
    clustered_df["cluster"],
    errors="coerce"
)


clustered_df = clustered_df.dropna(
    subset=["chapter#", "verse#", "cluster"]
).copy()

clustered_df["chapter#"] = clustered_df["chapter#"].astype(int)
clustered_df["verse#"] = clustered_df["verse#"].astype(int)
clustered_df["cluster"] = clustered_df["cluster"].astype(int)



if EXCLUDE_VERSE_ZERO:
    evaluation_df = clustered_df[
        clustered_df["verse#"] > 0
    ].copy()
else:
    evaluation_df = clustered_df.copy()

evaluation_df = evaluation_df.reset_index(drop=True)
evaluation_df.head(10)



,chapter#,verse#,verse,chapter,cluster
0,1,1,بسم الله الرحمٰن الرحيم,سورة الفاتحة,10
1,1,2,الحمد لله رب العٰلمين,سورة الفاتحة,16
2,1,3,الرحمٰن الرحيم,سورة الفاتحة,10
3,1,4,مٰلك يوم الدين,سورة الفاتحة,56
4,1,5,اياك نعبد واياك نستعين,سورة الفاتحة,0
5,1,6,اهدنا الصرٰط المستقيم,سورة الفاتحة,0
6,1,7,صرٰط الذين انعمت عليهم غير المغضوب عليهم ولا ا...,سورة الفاتحة,44
7,2,1,الم,سورة البقرة,55
8,2,2,ذٰلك الكتٰب لا ريب فيه هدى للمتقين,سورة البقرة,21
9,2,3,الذين يؤمنون بالغيب ويقيمون الصلوٰة ومما رزقنٰ...,سورة البقرة,48


In [13]:

# STRATIFIED SAMPLE ACROSS SURAHS


# Select the same number of verses from each Surah.
# This prevents the sample from being dominated by longer Surahs.

sampled_df = (
    evaluation_df
    .groupby("chapter#", group_keys=False, sort=False)
    .sample(
        n=N_VERSES_PER_SURAH,
        random_state=SAMPLE_RANDOM_STATE
    )
    .reset_index(drop=True)
)


sampled_df.head(10)

,chapter#,verse#,verse,chapter,cluster
0,1,1,بسم الله الرحمٰن الرحيم,سورة الفاتحة,10
1,2,85,ثم انتم هٰؤلاء تقتلون انفسكم وتخرجون فريقا منك...,سورة البقرة,40
2,3,96,ان اول بيت وضع للناس للذي ببكة مباركا وهدى للع...,سورة ال عمران,0
3,4,34,الرجال قوٰمون على النساء بما فضل الله بعضهم عل...,سورة النساء,67
4,5,115,قال الله اني منزلها عليكم فمن يكفر بعد منكم فا...,سورة المائدة,16
5,6,76,فلما جن عليه اليل رءا كوكبا قال هٰذا ربي فلما ...,سورة الانعام,26
6,7,58,والبلد الطيب يخرج نباته باذن ربه والذي خبث لا ...,سورة الاعراف,21
7,8,25,واتقوا فتنة لا تصيبن الذين ظلموا منكم خاصة واع...,سورة الانفال,43
8,9,115,وما كان الله ليضل قوما بعد اذ هدىٰهم حتىٰ يبين...,سورة التوبة,3
9,10,84,وقال موسىٰ يٰقوم ان كنتم ءامنتم بالله فعليه تو...,سورة يونس,1


In [14]:

# Count the number of verses in each saved cluster
cluster_sizes = (
    evaluation_df["cluster"]
    .value_counts()
    .rename("cluster_size")
)

# Add the relevant cluster size to every sampled verse
sampled_df = sampled_df.merge(
    cluster_sizes,
    left_on="cluster",
    right_index=True,
    how="left"
)

# The Streamlit app displays all verses in the same cluster.
# Exclude the queried verse itself from the returned list.
if EXCLUDE_QUERY_VERSE:
    sampled_df["result_list_length"] = (
        sampled_df["cluster_size"] - 1
    )
else:
    sampled_df["result_list_length"] = (
        sampled_df["cluster_size"]
    )

sampled_df.head(10)

,chapter#,verse#,verse,chapter,cluster,cluster_size,result_list_length
0,1,1,بسم الله الرحمٰن الرحيم,سورة الفاتحة,10,5,4
1,2,85,ثم انتم هٰؤلاء تقتلون انفسكم وتخرجون فريقا منك...,سورة البقرة,40,64,63
2,3,96,ان اول بيت وضع للناس للذي ببكة مباركا وهدى للع...,سورة ال عمران,0,875,874
3,4,34,الرجال قوٰمون على النساء بما فضل الله بعضهم عل...,سورة النساء,67,65,64
4,5,115,قال الله اني منزلها عليكم فمن يكفر بعد منكم فا...,سورة المائدة,16,52,51
5,6,76,فلما جن عليه اليل رءا كوكبا قال هٰذا ربي فلما ...,سورة الانعام,26,121,120
6,7,58,والبلد الطيب يخرج نباته باذن ربه والذي خبث لا ...,سورة الاعراف,21,121,120
7,8,25,واتقوا فتنة لا تصيبن الذين ظلموا منكم خاصة واع...,سورة الانفال,43,207,206
8,9,115,وما كان الله ليضل قوما بعد اذ هدىٰهم حتىٰ يبين...,سورة التوبة,3,164,163
9,10,84,وقال موسىٰ يٰقوم ان كنتم ءامنتم بالله فعليه تو...,سورة يونس,1,56,55


In [19]:


# DISPLAY EACH SAMPLED VERSE AND ITS LIST LENGTH

results_df = sampled_df[
    [
        "chapter#",
        "verse#",
        "verse",
        "cluster",
        "cluster_size",
        "result_list_length"
    ]
].sort_values(
    by=["chapter#", "verse#"]
).reset_index(drop=True)

display(results_df)


,chapter#,verse#,verse,cluster,cluster_size,result_list_length
0,1,1,بسم الله الرحمٰن الرحيم,10,5,4
1,2,85,ثم انتم هٰؤلاء تقتلون انفسكم وتخرجون فريقا منك...,40,64,63
2,3,96,ان اول بيت وضع للناس للذي ببكة مباركا وهدى للع...,0,875,874
3,4,34,الرجال قوٰمون على النساء بما فضل الله بعضهم عل...,67,65,64
4,5,115,قال الله اني منزلها عليكم فمن يكفر بعد منكم فا...,16,52,51
...,...,...,...,...,...,...
109,110,1,اذا جاء نصر الله والفتح,11,105,104
110,111,2,ما اغنىٰ عنه ماله وما كسب,15,38,37
111,112,2,الله الصمد,3,164,163
112,113,2,من شر ما خلق,5,43,42


In [20]:
results_df.to_csv('../data/sampled_kmeans.csv', index=False, encoding='utf-8-sig')

In [17]:

list_lengths = results_df["result_list_length"]

print(f"Number of sampled verses: {len(results_df)}")
print(f"Number of saved clusters: {evaluation_df['cluster'].nunique()}")
print(f"Mean result-list length: {list_lengths.mean():.2f}")
print(f"Median result-list length: {list_lengths.median():.2f}")
print(f"Minimum result-list length: {list_lengths.min()}")
print(f"Maximum result-list length: {list_lengths.max()}")

Number of sampled verses: 114
Number of saved clusters: 70
Mean result-list length: 262.16
Median result-list length: 106.50
Minimum result-list length: 4
Maximum result-list length: 874


In [21]:
import numpy as np

# Set seed
rng = np.random.default_rng(seed=42)

# Choose 20 distinct chapters from 1 to 114
selected_chapters = np.sort(rng.choice(np.arange(1, 115), size=20, replace=False))

print("Audit these chapters:", selected_chapters)

Audit these chapters: [  9  10  15  21  43  44  52  56  57  64  72  75  79  83  86  87  95 101
 104 109]


In [22]:
np.median([164,89,18,60,88,66,875,875,317,68,164,64,875,121,67,875,88,875])

np.float64(105.0)

In [23]:
np.median([0,0,4,0,4,6,3,1,5,1,5,7,0,0,0,0,1,0])

np.float64(1.0)